# Actuarial Data Validation Engine
This notebook contains the converted python code for the data validation engine using `pandas`, `pydantic`, and `agno` (Google Gemini).

It is split into different stages so that you can run it easily in Google Colab.

**Note:** Ensure you have added your API Key and Model Name in the Colab Secrets tab as `GEMINI_API_KEY` and `GEMINI_MODEL_NAME`.

In [ ]:
# Install Required Packages
!pip install pandas numpy pydantic agno google-genai openpyxl python-dateutil

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from datetime import datetime
from dateutil import parser
import json
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

from agno.agent import Agent
from agno.models.google import Gemini
from google.colab import userdata

# Set Gemini API Key and Model from Colab Secrets
try:

    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    gemini_model_name = userdata.get('GEMINI_MODEL')
except Exception as e:
    print("Warning: Ensure you are running in Colab and have set 'GEMINI_API_KEY' and 'GEMINI_MODEL_NAME' in secrets.")
    # Fallback to a default model if secret is missing
    gemini_model_name = "gemini-2.5-flash"

################################ Path ######################################
# Using Colab's default runtime environment path ('/content')
Path = "/content"
############################################################################


## Step 1: Initial setup, Models, Helper Functions

In [ ]:
################################ Models ####################################

# Model 1: Column Schema Describer Schema
class Column_str_describer_AI_output(BaseModel):
    File_name: Literal["sales_data", "claims_data"] = Field(alias="File name", description="Name of the target dataset.")
    Column_name: str = Field(alias="Column name", description="Exact name of the column as present in the dataset.")
    Data_type: Literal["String", "Number", "Date"] = Field(alias="Data type", description="Inferred statistical/R data type.")
    Description: str = Field(description="Elaborate business and technical description. Explain column meaning, potential relationship to other columns (e.g. Sale Date <= Claim Date), and key validation conditions needed downstream.")

class ColumnSchemaList(BaseModel):
    items: List[Column_str_describer_AI_output]

# Gemini AI client for dataset schema extraction
Column_str_describer_AI = Agent(
    model=Gemini(id=gemini_model_name),
    instructions="""
    You are an expert insurance data architect analyzing datasets for ABC General Insurance Company.
    For each attached dataset (e.g. vehicle sales, warranty claims):
    1. Inspect all columns, data types, sample values, and statistical distributions.
    2. Determine nullability, primary key constraints, and allowable domains/ranges.
    3. Formulate exhaustive column descriptions focusing on business context and inter-table/intra-table relationships (e.g., cross-column comparisons, foreign key lookups between sales and claims).
    """,
    output_schema=ColumnSchemaList
)

# Model 2: Validation Rule Generator Schema
class Validation_rules_AI_output(BaseModel):
    Rule_ID: str = Field(alias="Rule ID", description="Unique identifier for the rule e.g. R001, R002.")
    Table_1: Literal["sales_data", "claims_data"] = Field(alias="Table 1", description="Primary dataset to validate.")
    Column_1: str = Field(alias="Column 1", description="Primary column in Table 1 being validated.")
    Rule_type: Literal["missing", "unique", "positive", "range", "greater_than", "equals", "exists", "allowed_values"] = Field(alias="Rule type", description="Category of the data validation rule.")
    Table_2: Optional[str] = Field(alias="Table 2", description="Secondary/reference dataset for cross-table rules (e.g. sales_data). Leave empty/null for intra-table rules.")
    Column_2: Optional[str] = Field(alias="Column 2", description="Secondary column in Table 1 (or Table 2) for 'greater_than', 'equals', or 'exists' rules. Leave empty/null if not applicable.")
    Parameter: Optional[str] = Field(description="Required parameter. For 'allowed_values': comma-separated categories. For 'range': min and max values ('min, max'). Leave empty/null for others.")
    Severity: Literal["Error", "Warning"] = Field(description="Severity level if the rule fails.")
    Description: str = Field(description="Detailed explanation of what this rule validates and why.")

class ValidationRuleList(BaseModel):
    items: List[Validation_rules_AI_output]

# Gemini AI client for generating validation rules
Validation_rules_AI = Agent(
    model=Gemini(id=gemini_model_name),
    instructions="""
    You are an actuarial data validation expert for ABC General Insurance Company.
    Using the provided data schema and sample dataset values, generate an EXHAUSTIVE list of data validation rules (aim for 50+ total rules). But do not create stupid rules like setting ranges for VIN, Claim ID etc..
    Allowed rule types are strictly: missing, unique, positive, range, greater_than, equals, exists, allowed_values.

    CRITICAL MANDATORY INSTRUCTIONS:
    1. EVERY SINGLE COLUMN in both sales_data and claims_data MUST have multiple rules defined for it.
    2. For EVERY column, evaluate if a 'missing' (null check) rule should exist. Do NOT omit missing checks!
    3. For EVERY numeric column (prices, costs, IDs, years, counts), evaluate 'positive' and/or 'range' rules.
    4. For EVERY categorical/string column (status, state, model, claim type), evaluate 'allowed_values' and extract all unique values from sample data.
    5. For EVERY ID/Key column (Sale_ID, Claim_ID, VIN), define 'unique' in Table 1 and 'exists' (foreign key lookup in Table 2).
    6. For ALL date and numeric comparisons (e.g., Claim Date vs Sale Date, Repair Cost vs Claim Amount), use 'greater_than' (swap Column 1/Column 2 if checking less than) or 'equals'. If comparing across tables, specify Table 2.

    CRITICAL PARAMETER RULES:
    - For 'allowed_values': Provide a comma-separated list of all valid categories observed in the dataset.
    - For 'range': Provide exact numeric boundaries as 'min, max' (e.g., '0, 100000').
    - For 'missing', 'unique', 'positive', 'greater_than', 'equals', 'exists': Parameter should be left empty/null.

    Do NOT stop early. You MUST generate as many executable rules logically possible across all columns and relationships.
    """,
    output_schema=ValidationRuleList
)


In [ ]:
################################ Functions #################################

# Function: Date_converter_fxn
# Converts messy Excel serial numbers (e.g., 44629, NA, 13/08/2018 as text) or formatted strings into standardized Date objects
def Date_converter_fxn(date_col_to_fix):
    date_fixed = []

    for date_val in date_col_to_fix:
        if pd.isna(date_val):
            date_fixed.append(pd.NaT)
            continue

        date_str = str(date_val).strip()

        if date_str.lower() in ('nan', 'nat', 'na', ''):
            date_fixed.append(pd.NaT)
        elif len(date_str) == 5 and date_str.isdigit():
            # Excel serial number (origin is generally 1899-12-30)
            date_fixed.append(pd.to_datetime(int(date_str), unit='D', origin='1899-12-30'))
        else:
            try:
                date_fixed.append(pd.to_datetime(parser.parse(date_str)))
            except:
                date_fixed.append(pd.NaT)

    return pd.Series(date_fixed)


In [ ]:
################################ Validator functions ######################

# Helper: Key lookup for cross-table comparisons
# Matches key1 from primary table to key2 in reference table and retrieves target_col
def lookup_ref_column(key1, ref_key2, ref_target_col):
    ref_series = pd.Series(ref_target_col.values, index=ref_key2.values)
    return key1.map(ref_series)

# 1. Missing check (returns True where col1 is NA or blank)
def validate_missing(col1):
    return col1.isna() | (col1.astype(str).str.strip() == "") | (col1.astype(str).str.lower() == "nan")

# 2. Unique check (returns True for duplicate entries)
def validate_unique(col1):
    return col1.duplicated(keep=False) & ~col1.isna()

# 3. Positive check (returns True where col1 is non-null and <= 0)
def validate_positive(col1):
    col_num = pd.to_numeric(col1, errors='coerce')
    return ~col_num.isna() & (col_num <= 0)

# 4. Range check (returns True where col1 falls outside [min_val, max_val])
def validate_range(col1, min_val=-float('inf'), max_val=float('inf')):
    col_num = pd.to_numeric(col1, errors='coerce')
    return ~col_num.isna() & ((col_num < min_val) | (col_num > max_val))

# 5. Greater Than check (checks if col1 > col2, returns True where col1 <= col2)
def validate_greater_than(col1, col2):
    valid_mask = ~col1.isna() & ~col2.isna()
    try:
        violation = pd.to_numeric(col1, errors='coerce') <= pd.to_numeric(col2, errors='coerce')
        if violation.isna().all():
             violation = pd.to_datetime(col1, errors='coerce') <= pd.to_datetime(col2, errors='coerce')
    except:
        violation = col1 <= col2
    return valid_mask & violation

# 6. Equals check (checks if col1 == col2, returns True where col1 != col2)
def validate_equals(col1, col2):
    valid_mask = ~col1.isna() & ~col2.isna()
    violation = col1 != col2
    return valid_mask & violation

# 7. Exists / Foreign Key lookup check (returns True where col1 is missing from ref_col)
def validate_exists(col1, ref_col):
    return ~col1.isna() & ~col1.isin(ref_col)

# 8. Allowed values check (returns True where col1 is not in allowed_set, handling whitespace)
def validate_allowed_values(col1, allowed_set):
    clean_col1 = col1.astype(str).str.strip()
    clean_allowed = [str(x).strip() for x in allowed_set]
    return ~col1.isna() & ~clean_col1.isin(clean_allowed)

# Dictionary mapping rule type strings to functions
validator_tools = {
    "lookup_ref": lookup_ref_column,
    "missing": validate_missing,
    "unique": validate_unique,
    "positive": validate_positive,
    "range": validate_range,
    "greater_than": validate_greater_than,
    "equals": validate_equals,
    "exists": validate_exists,
    "allowed_values": validate_allowed_values
}


## Generate Data Schema

In [ ]:
################################ Data Schema generator #####################
os.makedirs(os.path.join(Path, "Output"), exist_ok=True)
os.makedirs(os.path.join(Path, "Input"), exist_ok=True)

### Raw input Excel files read in
# Upload your files directly to the /content/Input/ directory or update the paths below if uploaded elsewhere
file1 = pd.read_excel(os.path.join(Path, "Input", "Sales data.xlsx"))
file2 = pd.read_excel(os.path.join(Path, "Input", "Claims data.xlsx"))

print("Excel files loaded successfully")

### Query Gemini AI to analyze raw structure and generate Data_Schema
question = (
    "The attached are 2 data tables from ABC insurance company.\n"
    "Sales refers to all the vehicle sales made in the year,\n"
    "Claims refers to all the claims made in the year\n"
    f"Sales sample data: \n{file1.head(50).to_string()}\n"
    f"Claims sample data: \n{file2.head(50).to_string()}\n"
    f"Sales structure: \n{file1.dtypes.to_string()}\n"
    f"Claims structure: \n{file2.dtypes.to_string()}\n"
)

response = Column_str_describer_AI.run(question)
response_DF = pd.DataFrame([item.model_dump(by_alias=True) for item in response.content.items])

### Save Data_Schema to Output Excel
response_DF.to_excel(os.path.join(Path, "Output", "Data_Schema.xlsx"), index=False)
print("Data schema generated and saved successfully. Please review them carefully before proceeding.")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Excel files loaded successfully
Data schema generated and saved successfully. Please review them carefully before proceeding.


## Step 2: Data cleaning & Validation rules generator

In [ ]:
################################ Data cleaning #############################
### Load verified Data_Schema and perform data type standardization
Data_schema = pd.read_excel(os.path.join(Path, "Output", "Data_Schema.xlsx"))

sales_data = file1.copy()
claims_data = file2.copy()

our_datasets = {"sales_data": sales_data, "claims_data": claims_data}

# Convert dates and numbers according to schema definitions
for _, row in Data_schema.iterrows():
    col_type = row['Data type']
    file_name = row['File name']
    col_name = row['Column name']

    if file_name in our_datasets and col_name in our_datasets[file_name].columns:
        if col_type == "Date":
            our_datasets[file_name][col_name] = Date_converter_fxn(our_datasets[file_name][col_name])

        if col_type == "Number":
            our_datasets[file_name][col_name] = pd.to_numeric(our_datasets[file_name][col_name], errors='coerce')

print("Data cleaning complete")

################################ Validation rules generator ################
### Generate validation rules using Validation_rules_AI based on cleaned schema
Data_schema_json = Data_schema.to_json(orient='records', indent=2)

rules_question = (
    "Here is the verified data schema with descriptions for the datasets:\n"
    f"{Data_schema_json}\n\n"
    "Sales sample data:\n"
    f"{sales_data.head(50).to_string()}\n\n"
    "Claims sample data:\n"
    f"{claims_data.head(50).to_string()}\n\n"
)

rules_response = Validation_rules_AI.run(rules_question)
validation_rules_DF = pd.DataFrame([item.model_dump(by_alias=True) for item in rules_response.content.items])

### Export generated validation rules to Output/Validation_Rules.xlsx
validation_rules_DF.to_excel(os.path.join(Path, "Output", "Validation_Rules.xlsx"), index=False)
print("Validation rules generated and saved successfully. Please review them carefully before proceeding.")


## Step 3: Validation rules engine

In [ ]:
################################ Validation rules engine ###################

### Deterministic Rule Execution Engine
# Reads Validation_Rules.xlsx and executes corresponding python validator functions
# Initialize validation report container
validation_report_list = []

try:
    validation_rules_to_run = pd.read_excel(os.path.join(Path, "Output", "Validation_Rules.xlsx"))
except FileNotFoundError:
    print("Validation_Rules.xlsx not found.")
    validation_rules_to_run = pd.DataFrame()

if not validation_rules_to_run.empty:
    for _, rule in validation_rules_to_run.iterrows():
        rule_id   = rule.get('Rule ID')
        t1_name   = rule.get('Table 1')
        col1_name = rule.get('Column 1')
        rule_type = rule.get('Rule type')
        t2_name   = rule.get('Table 2')
        col2_name = rule.get('Column 2')
        param     = rule.get('Parameter')
        severity  = rule.get('Severity')
        desc      = rule.get('Description')

        # Target dataset and primary column vector
        if t1_name not in our_datasets or col1_name not in our_datasets[t1_name].columns:
            continue

        df1  = our_datasets[t1_name]
        col1 = df1[col1_name]

        violation_mask = pd.Series([False] * len(df1), index=df1.index)
        col2 = None

        # Dispatch to appropriate validator function based on rule_type
        if rule_type == "missing":
            violation_mask = validator_tools["missing"](col1)
        elif rule_type == "unique":
            violation_mask = validator_tools["unique"](col1)
        elif rule_type == "positive":
            violation_mask = validator_tools["positive"](col1)
        elif rule_type == "allowed_values":
            # Expect comma-separated string e.g. "Completed, Pending"
            if pd.notna(param):
                allowed_set = [x.strip() for x in str(param).split(",")]
                violation_mask = validator_tools["allowed_values"](col1, allowed_set)
        elif rule_type == "range":
            # Expect parameter like "0, 100" or "[0, 100]"
            if pd.notna(param):
                num_params = [float(x) for x in re.findall(r'-?\d+\.?\d*', str(param))]
                min_v = num_params[0] if len(num_params) >= 1 else -float('inf')
                max_v = num_params[1] if len(num_params) >= 2 else float('inf')
                violation_mask = validator_tools["range"](col1, min_val=min_v, max_val=max_v)
        elif rule_type == "exists":
            if pd.notna(t2_name) and pd.notna(col2_name) and t2_name in our_datasets and col2_name in our_datasets[t2_name].columns:
                ref_vec = our_datasets[t2_name][col2_name]
                violation_mask = validator_tools["exists"](col1, ref_vec)
        elif rule_type in ["greater_than", "equals"]:
            # Resolve Column 2 (same-table or cross-table lookup via lookup_ref)
            if pd.notna(t2_name) and str(t2_name).strip() != "" and t2_name != t1_name:
                if t2_name in our_datasets and col1_name in our_datasets[t2_name].columns and col2_name in our_datasets[t2_name].columns:
                    key1 = df1[col1_name]
                    t2_key2 = our_datasets[t2_name][col1_name]
                    t2_target = our_datasets[t2_name][col2_name]

                    col2 = validator_tools["lookup_ref"](key1, t2_key2, t2_target)
            else:
                if pd.notna(col2_name) and col2_name in df1.columns:
                    col2 = df1[col2_name]

            if col2 is not None:
                if rule_type == "greater_than":
                    violation_mask = validator_tools["greater_than"](col1, col2)
                elif rule_type == "equals":
                    violation_mask = validator_tools["equals"](col1, col2)

        # Log violations directly
        violated_rows = df1.index[violation_mask].tolist()

        if len(violated_rows) > 0:
            # Extract VIN if present in df1
            vin_col = df1.loc[violated_rows, "VIN"].astype(str).tolist() if "VIN" in df1.columns else [pd.NA] * len(violated_rows)

            # Value for Column 1
            val1 = col1.loc[violated_rows].astype(str).tolist()

            # Value for Column 2
            if rule_type in ["greater_than", "equals"] and col2 is not None:
                val2 = col2.loc[violated_rows].astype(str).tolist()
            else:
                val2 = [pd.NA] * len(violated_rows)

            col2_name_val = col2_name if pd.notna(col2_name) else ""

            for i, row_idx in enumerate(violated_rows):
                err_dict = {
                    "Rule ID": rule_id,
                    "Table 1": t1_name,
                    "VIN": vin_col[i],
                    "Row Index": row_idx + 1, # 1-indexed to match R
                    "Column 1": col1_name,
                    "Value 1": val1[i],
                    "Column 2": col2_name_val,
                    "Value 2": val2[i],
                    "Severity": severity,
                    "Description": desc
                }
                validation_report_list.append(err_dict)

### Fallback if no violations are found
if len(validation_report_list) == 0:
    validation_report = pd.DataFrame([{"Message": "No validation errors found."}])
else:
    validation_report = pd.DataFrame(validation_report_list)

# Write final report to Excel
validation_report.to_excel(os.path.join(Path, "Output", "Validation_Report.xlsx"), index=False)
print("Validation execution complete. Report saved to Output/Validation_Report.xlsx")